In [1]:
! pip3 install --upgrade --quiet  google-cloud-aiplatform

In [2]:
PROJECT_ID = "project-eac74fb9-0e15-492a-ab1"
LOCATION = "us-central1"

In [3]:
BUCKET_NAME = "mlops-course-project-eac74fb9-0e15-492a-ab1-v4-unique"
BUCKET_URI = f"gs://{BUCKET_NAME}"

In [4]:
MODEL_ARTIFACT_DIR="iris_classifier/model"

In [5]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [6]:
import os
import sys

In [7]:
import pandas as pd

In [8]:
df = pd.read_csv("data.csv")

In [9]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.8,4.0,1.2,0.2,setosa
1,5.7,4.4,1.5,0.4,setosa
2,5.4,3.9,1.3,0.4,setosa
3,5.1,3.5,1.4,0.3,setosa
4,5.7,3.8,1.7,0.3,setosa


# Dataset Preparation & Cloud Upload

In [10]:
from google.cloud import storage

In [11]:
client = storage.Client(project=PROJECT_ID)

In [12]:
bucket = client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = client.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"✅ Bucket created: {BUCKET_NAME}")
else:
    print(f"✅ Bucket already exists: {BUCKET_NAME}")

✅ Bucket already exists: mlops-course-project-eac74fb9-0e15-492a-ab1-v4-unique


In [13]:
from sklearn.model_selection import train_test_split

In [14]:
train, test = train_test_split(df, test_size = 0.4, stratify = df['species'], random_state = 42)

In [ ]:
# Split and upload train/eval
train.to_csv("iris_train.csv", index=False)
test.to_csv("iris_eval.csv", index=False)

In [16]:
bucket.blob("data/iris_train.csv").upload_from_filename("iris_train.csv")
bucket.blob("data/iris_eval.csv").upload_from_filename("iris_eval.csv")

In [17]:
print("✅ Train and eval data uploaded to GCS!")
print(f"   → gs://{BUCKET_NAME}/data/iris_train.csv")
print(f"   → gs://{BUCKET_NAME}/data/iris_eval.csv")

✅ Train and eval data uploaded to GCS!
   → gs://mlops-course-project-eac74fb9-0e15-492a-ab1-v4-unique/data/iris_train.csv
   → gs://mlops-course-project-eac74fb9-0e15-492a-ab1-v4-unique/data/iris_eval.csv


# Training Pipeline

In [18]:
import pickle
import json
from datetime import datetime
from google.cloud import storage

In [19]:
TIMESTAMP = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
ARTIFACT_DIR = f"artifacts/{TIMESTAMP}"
print(f"🕐 Run timestamp: {TIMESTAMP}")

🕐 Run timestamp: 2026-06-23T05-04-58


In [20]:
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

In [21]:
bucket.blob("data/iris_train.csv").download_to_filename("iris_train.csv")
bucket.blob("data/iris_eval.csv").download_to_filename("iris_eval.csv")

In [22]:
train_df = pd.read_csv("iris_train.csv")
eval_df  = pd.read_csv("iris_eval.csv")

In [23]:
X_train = train_df[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train_df['species']
X_eval  = eval_df[['sepal_length','sepal_width','petal_length','petal_width']]
y_eval  = eval_df['species']

In [25]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

In [26]:
model = DecisionTreeClassifier(max_depth=3, random_state=1)
model.fit(X_train, y_train)
print("✅ Model trained!")

✅ Model trained!


In [27]:
# Making Predictions

predictions = model.predict(X_eval)
accuracy = metrics.accuracy_score(y_eval, predictions)
print(f"✅ Accuracy: {accuracy:.3f}")

✅ Accuracy: 0.951


In [28]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

In [29]:
log = {
    "timestamp": TIMESTAMP,
    "accuracy": accuracy,
    "model": "DecisionTreeClassifier",
    "max_depth": 3,
    "train_size": len(train_df),
    "eval_size": len(eval_df)
}

In [30]:
with open("training_log.json", "w") as f:
    json.dump(log, f, indent=2)

In [31]:
bucket.blob(f"{ARTIFACT_DIR}/model.pkl").upload_from_filename("model.pkl")
bucket.blob(f"{ARTIFACT_DIR}/training_log.json").upload_from_filename("training_log.json")

In [32]:
print(f"✅ Artifacts saved to gs://{BUCKET_NAME}/{ARTIFACT_DIR}/")
print(f"   → model.pkl")
print(f"   → training_log.json")

✅ Artifacts saved to gs://mlops-course-project-eac74fb9-0e15-492a-ab1-v4-unique/artifacts/2026-06-23T05-04-58/
   → model.pkl
   → training_log.json


In [33]:
LATEST_ARTIFACT_DIR = ARTIFACT_DIR

# INFERENCE

In [34]:
print(f"📦 Loading model from: gs://{BUCKET_NAME}/{LATEST_ARTIFACT_DIR}/model.pkl")
bucket.blob(f"{LATEST_ARTIFACT_DIR}/model.pkl").download_to_filename("model_loaded.pkl")

📦 Loading model from: gs://mlops-course-project-eac74fb9-0e15-492a-ab1-v4-unique/artifacts/2026-06-23T05-04-58/model.pkl


In [35]:
with open("model_loaded.pkl", "rb") as f:
    loaded_model = pickle.load(f)

In [36]:
bucket.blob("data/iris_eval.csv").download_to_filename("iris_eval.csv")
eval_df = pd.read_csv("iris_eval.csv")

In [37]:
X_eval = eval_df[['sepal_length','sepal_width','petal_length','petal_width']]
y_eval = eval_df['species']

In [38]:
predictions = loaded_model.predict(X_eval)
accuracy = metrics.accuracy_score(y_eval, predictions)

In [39]:
print("✅ Inference complete!")
print(f"   Accuracy on eval set: {accuracy:.3f}")

✅ Inference complete!
   Accuracy on eval set: 0.951


In [40]:
results = eval_df.copy()
results['predicted'] = predictions
print(results[['species','predicted']].head(10).to_string())

      species   predicted
0   virginica   virginica
1      setosa      setosa
2  versicolor  versicolor
3   virginica   virginica
4  versicolor  versicolor
5      setosa      setosa
6   virginica   virginica
7  versicolor  versicolor
8  versicolor  versicolor
9      setosa      setosa


# SECOND RUN

In [41]:
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)
blobs = client.list_blobs(BUCKET_NAME, prefix="artifacts/")
for blob in blobs:
    print(blob.name)

artifacts/2026-06-22T17-35-00/model.pkl
artifacts/2026-06-22T17-35-00/training_log.json
artifacts/2026-06-22T17-40-45/model.pkl
artifacts/2026-06-22T17-40-45/training_log.json
artifacts/2026-06-22T17-44-55/model.pkl
artifacts/2026-06-22T17-44-55/training_log.json
artifacts/2026-06-23T05-04-58/model.pkl
artifacts/2026-06-23T05-04-58/training_log.json
